In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/clean_transactions.csv", parse_dates=["date"])

# Basic checks
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print(df["category"].value_counts())
print(df["txn_type"].value_counts())

# Separate expenses from income — you'll use this split constantly
expenses = df[df["txn_type"] == "debit"].copy()
income   = df[df["txn_type"] == "credit"].copy()
print(f"\nExpense rows: {len(expenses)}  |  Income rows: {len(income)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Category breakdown — donut chart (this goes straight into your dashboard later)
cat_totals = expenses.groupby("category")["amount"].sum().sort_values(ascending=False)
axes[0].pie(cat_totals, labels=cat_totals.index, autopct="%1.1f%%",
            startangle=90, wedgeprops=dict(width=0.5))
axes[0].set_title("Spend by category (donut)")

# Amount distribution — log scale to handle skew
axes[1].hist(np.log1p(expenses["amount"]), bins=40, edgecolor="white")
axes[1].set_xlabel("log(amount)")
axes[1].set_title("Amount distribution (log scale)")

plt.tight_layout()
plt.savefig("../data/charts/spend_distribution.png", dpi=150)
plt.show()

# Monthly spend over time — this is your forecast baseline
monthly = (
    expenses
    .groupby(expenses["date"].dt.to_period("M"))["amount"]
    .sum()
    .reset_index()
)
monthly["date"] = monthly["date"].dt.to_timestamp()

plt.figure(figsize=(12, 4))
plt.plot(monthly["date"], monthly["amount"], marker="o", linewidth=2)
plt.fill_between(monthly["date"], monthly["amount"], alpha=0.1)
plt.title("Total monthly spend")
plt.ylabel("Amount (INR)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../data/charts/monthly_trend.png", dpi=150)
plt.show()

# Per-category monthly heatmap
cat_monthly = (
    expenses
    .groupby([expenses["date"].dt.to_period("M"), "category"])["amount"]
    .sum()
    .unstack(fill_value=0)
)
plt.figure(figsize=(14, 7))
sns.heatmap(cat_monthly.T, fmt=".0f", annot=True, cmap="YlOrRd", linewidths=0.5)
plt.title("Monthly spend per category")
plt.tight_layout()
plt.savefig("../data/charts/category_heatmap.png", dpi=150)
plt.show()

# Select the engineered features that will go into clustering
feature_cols = [
    "amount", "log_amount", "day_of_week", "is_weekend",
    "month", "quarter", "is_month_end",
    "rolling_7d_spend", "rolling_30d_spend",
    "monthly_cat_spend", "monthly_total_spend", "spend_ratio",
    "txns_that_day", "amount_vs_cat_median", "is_high_spend_day"
]

feat_df = expenses[feature_cols].dropna()

# Correlation matrix — spot multicollinearity before clustering
plt.figure(figsize=(12, 10))
corr = feat_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", center=0, linewidths=0.5, vmin=-1, vmax=1)
plt.title("Feature correlation matrix")
plt.tight_layout()
plt.savefig("../data/charts/correlation_matrix.png", dpi=150)
plt.show()

# Check for high correlations (> 0.85) — these cause problems in K-Means
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i):
        if abs(corr.iloc[i, j]) > 0.85:
            high_corr.append((corr.columns[i], corr.columns[j], corr.iloc[i, j]))
            
if high_corr:
    print("High correlation pairs (consider dropping one):")
    for a, b, v in high_corr:
        print(f"  {a}  <->  {b}  :  {v:.2f}")
else:
    print("No problematic correlations found.")

# Weekend vs weekday spend pattern
day_spend = (
    expenses.groupby("day_name")["amount"]
    .agg(["mean", "sum", "count"])
    .reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])
)
print(day_spend)

day_spend["mean"].plot(kind="bar", figsize=(9, 4), color="steelblue", edgecolor="white")
plt.title("Average transaction amount by day of week")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("../data/charts/day_of_week.png", dpi=150)
plt.show()

# Review anomaly flags
anomalies = expenses[expenses["is_high_spend_day"] == 1]
print(f"\nAnomaly transactions: {len(anomalies)}")
print(anomalies[["date", "description", "category", "amount", "rolling_30d_spend"]].head(10))